<a href="https://colab.research.google.com/github/sameerkarur/Data_science/blob/main/07_IITK_AIML_Capstone/project3_preserving_heritage/lms_upload/preserving_heritage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course 7 Capstone Project 3 — Preserving Heritage: Enhancing Tourism with AI

**Part 1:** Historical structure image classification with TensorFlow transfer learning (MobileNetV2)\
**Part 2:** Indonesian tourism EDA + collaborative filtering recommender

This notebook trains on a **stratified subset (~350 images/class)** for practical CPU runtime, evaluates on the full provided test set, compares training **without** vs **with** augmentation, and builds a place recommender from user ratings.


In [1]:
# Setup
import os, json, random, warnings
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("MPLCONFIGDIR", str(Path("outputs/.mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

ROOT = Path(".").resolve()
OUT = ROOT / "outputs"
MODELS = ROOT / "models"
OUT.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("Devices:", tf.config.list_physical_devices())
print("ROOT:", ROOT)


TensorFlow: 2.21.0
OpenCV: 5.0.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
ROOT: /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage


## PART 1 — Historical Structure Classification

### 1.1 Paths, classes, and sample counts


In [2]:
TRAIN_DIR = ROOT / "data/part1_clean/train"
TEST_DIR = ROOT / "data/part1_clean/test"
# Original full paths (for documentation / sample plots fallback)
TRAIN_FULL = ROOT / "data/part1/dataset_hist_structures 2/dataset_hist_structures/Stuctures_Dataset"
TEST_FULL = ROOT / "data/part1/dataset_hist_structures 2/dataset_hist_structures/Dataset_test/Dataset_test_original_1478"

IMG_SIZE = 160
BATCH = 32
VAL_ACC_TARGET = 0.85
MAX_PER_CLASS = 350  # documented stratified subset
# Images re-encoded to clean RGB JPEGs under data/part1_clean (TF-safe)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(base: Path):
    rows = []
    classes = sorted([p.name for p in base.iterdir() if p.is_dir() and not p.name.startswith(".")])
    for c in classes:
        n = sum(1 for p in (base / c).iterdir() if p.suffix.lower() in EXTS and p.is_file())
        rows.append({"class": c, "count": n})
    return pd.DataFrame(rows)

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
print("Train dir:", TRAIN_DIR)
print(train_counts.to_string(index=False))
print("\nTest dir:", TEST_DIR)
print(test_counts.to_string(index=False))
print("\nTrain total:", int(train_counts["count"].sum()), "| Test total:", int(test_counts["count"].sum()))
print(f"Note: training uses stratified subset up to {MAX_PER_CLASS} images/class for CPU practicality.")

train_counts.to_csv(OUT / "train_class_counts.csv", index=False)
test_counts.to_csv(OUT / "test_class_counts.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(train_counts))
ax.bar(x - 0.2, train_counts["count"], 0.4, label="Train (subset)")
ax.bar(x + 0.2, test_counts.set_index("class").reindex(train_counts["class"])["count"].values, 0.4, label="Test")
ax.set_xticks(x)
ax.set_xticklabels(train_counts["class"], rotation=45, ha="right")
ax.set_ylabel("Images")
ax.set_title("Samples per class (train subset vs test)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "class_counts.png", dpi=140)
plt.show()


Train dir: /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/data/part1_clean/train
          class  count
          altar    350
           apse    350
     bell_tower    350
         column    350
    dome(inner)    350
    dome(outer)    350
flying_buttress    350
       gargoyle    350
  stained_glass    350
          vault    350

Test dir: /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/data/part1_clean/test
          class  count
          altar    140
           apse     57
     bell_tower    170
         column    210
    dome(inner)     86
    dome(outer)    168
flying_buttress     78
       gargoyle    238
  stained_glass    162
          vault    164

Train total: 3500 | Test total: 1473
Note: training uses stratified subset up to 350 images/class for CPU practicality.


### 1.2 Sample images — 8–10 per class (OpenCV + matplotlib)


In [3]:
CLASSES = train_counts["class"].tolist()
N_SAMPLES = 9  # 8–10 range

def load_bgr_rgb(path, size=None):
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if size:
        img = cv2.resize(img, (size, size))
    return img

# Grid: one figure per class with N_SAMPLES thumbnails
for cls in CLASSES:
    folder = TRAIN_DIR / cls
    files = [p for p in folder.iterdir() if p.suffix.lower() in EXTS]
    random.shuffle(files)
    files = files[:N_SAMPLES]
    cols = 3
    rows = int(np.ceil(len(files) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(9, 3 * rows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(files):
            im = load_bgr_rgb(files[i], size=160)
            if im is not None:
                ax.imshow(im)
                ax.set_title(files[i].name[:18], fontsize=8)
    fig.suptitle(f"Class: {cls} (n={N_SAMPLES})", fontsize=12)
    plt.tight_layout()
    safe = cls.replace("(", "").replace(")", "").replace(" ", "_")
    fig.savefig(OUT / f"samples_{safe}.png", dpi=120)
    plt.show()
    plt.close(fig)

print("Saved per-class sample grids to outputs/samples_*.png")


Saved per-class sample grids to outputs/samples_*.png


### 1.3–1.5 Build MobileNetV2 transfer model, compile, and accuracy callback


In [4]:
NUM_CLASSES = len(CLASSES)
AUTOTUNE = tf.data.AUTOTUNE

def build_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout=0.3):
    inputs = keras.Input(shape=(img_size, img_size, 3))
    # Expect float in [0, 255]; MobileNetV2 preprocess scales to [-1, 1]
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    base = keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False  # freeze conv layers initially
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="heritage_mobilenetv2")
    return model, base

model, base_model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

class StopAtValAccuracy(keras.callbacks.Callback):
    """Stop when val accuracy hits threshold, but only after min_epochs (for useful curves)."""
    def __init__(self, threshold=0.85, min_epochs=3):
        super().__init__()
        self.threshold = threshold
        self.min_epochs = min_epochs
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_accuracy")
        # epoch is 0-indexed
        if val_acc is not None and val_acc >= self.threshold and (epoch + 1) >= self.min_epochs:
            print(f"\nReached val_accuracy={val_acc:.4f} >= {self.threshold} after {epoch+1} epochs. Stopping.")
            self.model.stop_training = True

stop_cb = StopAtValAccuracy(VAL_ACC_TARGET, min_epochs=3)
ckpt_path = MODELS / "best_heritage_classifier.keras"
ckpt_cb = keras.callbacks.ModelCheckpoint(
    str(ckpt_path), monitor="val_accuracy", save_best_only=True, mode="max", verbose=1
)
reduce_cb = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)
print("Callbacks ready. Target val accuracy:", VAL_ACC_TARGET)


Model: "heritage_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,588,490 (9.87 MB)

 Trainable params: 330,506 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

Callbacks ready. Target val accuracy: 0.85


### 1.6 Datasets (train / validation=test)


In [5]:
def make_dataset(directory, shuffle=True, augment=False):
    ds = keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH,
        shuffle=shuffle,
        seed=42,
    )
    if augment:
        aug = keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.08),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
        ], name="augmentation")
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(TRAIN_DIR, shuffle=True, augment=False)
val_ds = make_dataset(TEST_DIR, shuffle=False, augment=False)
print("Class names:", train_ds.class_names if hasattr(train_ds, 'class_names') else CLASSES)
# image_dataset_from_directory returns Dataset; class_names on the raw ds before map
# Capture from a fresh call:
_probe = keras.utils.image_dataset_from_directory(TRAIN_DIR, class_names=CLASSES, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)
print("Confirmed classes:", _probe.class_names)
del _probe


Found 3500 files belonging to 10 classes.


Found 1473 files belonging to 10 classes.


Class names: ['altar', 'apse', 'bell_tower', 'column', 'dome(inner)', 'dome(outer)', 'flying_buttress', 'gargoyle', 'stained_glass', 'vault']
Found 3500 files belonging to 10 classes.


Confirmed classes: ['altar', 'apse', 'bell_tower', 'column', 'dome(inner)', 'dome(outer)', 'flying_buttress', 'gargoyle', 'stained_glass', 'vault']


### 1.7 Train WITHOUT augmentation


In [6]:
EPOCHS_NO_AUG = 5

history_no_aug = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_NO_AUG,
    callbacks=[stop_cb, ckpt_cb, reduce_cb],
    verbose=1,
)

no_aug_best = float(max(history_no_aug.history.get("val_accuracy", [0])))
no_aug_last = float(history_no_aug.history["val_accuracy"][-1])
print(f"No-aug best val_accuracy: {no_aug_best:.4f} | last: {no_aug_last:.4f}")


Epoch 1/5


  1/110 ━━━━━━━━━━━━━━━━━━━━ 4:23 2s/step - accuracy: 0.0625 - loss: 3.4327

  2/110 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.0938 - loss: 3.0384

  3/110 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.0938 - loss: 2.8295

  4/110 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.1328 - loss: 2.6238

...

109/110 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.9627 - loss: 0.1047


Reached val_accuracy=0.9253 >= 0.85 after 3 epochs. Stopping.

Epoch 3: val_accuracy improved from 0.92261 to 0.92532, saving model to /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/models/best_heritage_classifier.keras



Epoch 3: finished saving model to /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/models/best_heritage_classifier.keras


110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.9629 - loss: 0.1045 - val_accuracy: 0.9253 - val_loss: 0.2249 - learning_rate: 0.0010


No-aug best val_accuracy: 0.9253 | last: 0.9253


### 1.8 Train WITH augmentation (continue / fresh head optional)

We rebuild the model head path with the same frozen backbone and train on an augmented pipeline to compare generalization.


In [7]:
# Fresh model for fair comparison with augmentation
model_aug, base_aug = build_model()
model_aug.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

train_ds_aug = make_dataset(TRAIN_DIR, shuffle=True, augment=True)
val_ds2 = make_dataset(TEST_DIR, shuffle=False, augment=False)

ckpt_aug = MODELS / "best_heritage_classifier_aug.keras"
ckpt_cb_aug = keras.callbacks.ModelCheckpoint(
    str(ckpt_aug), monitor="val_accuracy", save_best_only=True, mode="max", verbose=1
)
stop_cb2 = StopAtValAccuracy(VAL_ACC_TARGET, min_epochs=3)

EPOCHS_AUG = 6
history_aug = model_aug.fit(
    train_ds_aug,
    validation_data=val_ds2,
    epochs=EPOCHS_AUG,
    callbacks=[stop_cb2, ckpt_cb_aug, reduce_cb],
    verbose=1,
)

aug_best = float(max(history_aug.history.get("val_accuracy", [0])))
aug_last = float(history_aug.history["val_accuracy"][-1])
print(f"Aug best val_accuracy: {aug_best:.4f} | last: {aug_last:.4f}")


Found 3500 files belonging to 10 classes.


Found 1473 files belonging to 10 classes.


Epoch 1/6


  1/110 ━━━━━━━━━━━━━━━━━━━━ 4:38 3s/step - accuracy: 0.0625 - loss: 3.0145

  2/110 ━━━━━━━━━━━━━━━━━━━━ 11s 103ms/step - accuracy: 0.0469 - loss: 2.9731

...

109/110 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.9160 - loss: 0.2517 


Reached val_accuracy=0.9138 >= 0.85 after 3 epochs. Stopping.

Epoch 3: val_accuracy improved from 0.90292 to 0.91378, saving model to /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/models/best_heritage_classifier_aug.keras



Epoch 3: finished saving model to /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/models/best_heritage_classifier_aug.keras


110/110 ━━━━━━━━━━━━━━━━━━━━ 16s 140ms/step - accuracy: 0.9163 - loss: 0.2509 - val_accuracy: 0.9138 - val_loss: 0.2636 - learning_rate: 0.0010


Aug best val_accuracy: 0.9138 | last: 0.9138


### 1.9 Accuracy curves + save best model


In [8]:
def plot_history(hist, title, path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(hist.history["accuracy"], label="train")
    axes[0].plot(hist.history["val_accuracy"], label="val")
    axes[0].axhline(VAL_ACC_TARGET, color="gray", ls="--", lw=1, label=f"target {VAL_ACC_TARGET}")
    axes[0].set_title(f"{title} — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(hist.history["loss"], label="train")
    axes[1].plot(hist.history["val_loss"], label="val")
    axes[1].set_title(f"{title} — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(path, dpi=140)
    plt.show()
    plt.close(fig)

plot_history(history_no_aug, "No Augmentation", OUT / "curves_no_aug.png")
plot_history(history_aug, "With Augmentation", OUT / "curves_with_aug.png")

# Pick overall best checkpoint
best_path = ckpt_aug if aug_best >= no_aug_best else ckpt_path
# Also copy/save a canonical best
best_model = keras.models.load_model(best_path)
final_model_path = MODELS / "heritage_structure_classifier_best.keras"
best_model.save(final_model_path)

# Evaluate best on test
eval_loss, eval_acc = best_model.evaluate(val_ds2, verbose=0)
metrics = {
    "img_size": IMG_SIZE,
    "batch": BATCH,
    "max_per_class_train_subset": MAX_PER_CLASS,
    "num_classes": NUM_CLASSES,
    "classes": CLASSES,
    "no_aug_best_val_accuracy": no_aug_best,
    "aug_best_val_accuracy": aug_best,
    "final_best_val_accuracy": float(eval_acc),
    "final_best_val_loss": float(eval_loss),
    "improvement_aug_minus_no_aug": float(aug_best - no_aug_best),
    "best_checkpoint": str(best_path.name),
    "saved_model": str(final_model_path.name),
}
with open(OUT / "part1_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))
print("Saved best model to", final_model_path)


{
  "img_size": 160,
  "batch": 32,
  "max_per_class_train_subset": 350,
  "num_classes": 10,
  "classes": [
    "altar",
    "apse",
    "bell_tower",
    "column",
    "dome(inner)",
    "dome(outer)",
    "flying_buttress",
    "gargoyle",
    "stained_glass",
    "vault"
  ],
  "no_aug_best_val_accuracy": 0.9253224730491638,
  "aug_best_val_accuracy": 0.9137814044952393,
  "final_best_val_accuracy": 0.9253224730491638,
  "final_best_val_loss": 0.2249237298965454,
  "improvement_aug_minus_no_aug": -0.01154106855392456,
  "best_checkpoint": "best_heritage_classifier.keras",
  "saved_model": "heritage_structure_classifier_best.keras"
}
Saved best model to /Users/sameerkarur/Documents/Git/Data_science/07_IITK_AIML_Capstone/project3_preserving_heritage/models/heritage_structure_classifier_best.keras


## PART 2 — Tourism EDA + Collaborative Filtering Recommender


In [9]:
PART2 = ROOT / "data/part2"
users = pd.read_csv(PART2 / "user.csv")
ratings = pd.read_csv(PART2 / "tourism_rating.csv")
places = pd.read_excel(PART2 / "tourism_with_id.xlsx")

print("users", users.shape, list(users.columns))
print("ratings", ratings.shape, list(ratings.columns))
print("places", places.shape, list(places.columns))

# Drop unnamed junk columns
places = places.loc[:, ~places.columns.astype(str).str.startswith("Unnamed")].copy()
print("places cleaned columns:", list(places.columns))


users (300, 3) ['User_Id', 'Location', 'Age']
ratings (10000, 3) ['User_Id', 'Place_Id', 'Place_Ratings']
places (437, 13) ['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Unnamed: 11', 'Unnamed: 12']
places cleaned columns: ['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long']


### 2.1 Missing values & duplicates cleanup


In [10]:
def report_clean(name, df, subset_cols=None):
    print(f"\n=== {name} ===")
    print("missing:\n", df.isna().sum())
    dup = df.duplicated().sum()
    print("exact duplicate rows:", int(dup))
    if subset_cols:
        d2 = df.duplicated(subset=subset_cols).sum()
        print(f"duplicates on {subset_cols}:", int(d2))

report_clean("users", users, ["User_Id"])
report_clean("ratings", ratings, ["User_Id", "Place_Id"])
report_clean("places", places, ["Place_Id"])

users_c = users.drop_duplicates(subset=["User_Id"]).copy()
ratings_c = ratings.drop_duplicates(subset=["User_Id", "Place_Id"], keep="last").copy()
places_c = places.drop_duplicates(subset=["Place_Id"]).copy()

# Fill sparse numeric Time_Minutes with median if present
if "Time_Minutes" in places_c.columns:
    places_c["Time_Minutes"] = places_c["Time_Minutes"].fillna(places_c["Time_Minutes"].median())

print("\nAfter cleanup:", users_c.shape, ratings_c.shape, places_c.shape)



=== users ===
missing:
 User_Id     0
Location    0
Age         0
dtype: int64
exact duplicate rows: 0
duplicates on ['User_Id']: 0

=== ratings ===
missing:
 User_Id          0
Place_Id         0
Place_Ratings    0
dtype: int64
exact duplicate rows: 79
duplicates on ['User_Id', 'Place_Id']: 403

=== places ===
missing:
 Place_Id          0
Place_Name        0
Description       0
Category          0
City              0
Price             0
Rating            0
Time_Minutes    232
Coordinate        0
Lat               0
Long              0
dtype: int64
exact duplicate rows: 0
duplicates on ['Place_Id']: 0

After cleanup: (300, 3) (9597, 3) (437, 11)


### 2.2 Age distribution & where tourists come from


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(users_c["Age"], bins=15, edgecolor="white")
axes[0].set_title("Tourist Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

# Location is "City, Province" style
loc_counts = users_c["Location"].value_counts().head(12)
axes[1].barh(loc_counts.index[::-1], loc_counts.values[::-1])
axes[1].set_title("Top Tourist Origins (Location)")
axes[1].set_xlabel("Users")
plt.tight_layout()
plt.savefig(OUT / "eda_age_origins.png", dpi=140)
plt.show()

print("Age describe:\n", users_c["Age"].describe())
print("\nTop origins:\n", loc_counts.head(10))


Age describe:
 count    300.000000
mean      28.700000
std        6.393716
min       18.000000
25%       24.000000
50%       29.000000
75%       34.000000
max       40.000000
Name: Age, dtype: float64

Top origins:
 Location
Bekasi, Jawa Barat              39
Semarang, Jawa Tengah           22
Lampung, Sumatera Selatan       20
Yogyakarta, DIY                 20
Bogor, Jawa Barat               17
Cirebon, Jawa Barat             14
Jakarta Selatan, DKI Jakarta    14
Subang, Jawa Barat              14
Depok, Jawa Barat               12
Ponorogo, Jawa Timur            11
Name: count, dtype: int64


### 2.3 Spot categories, tourism by location, best city for nature enthusiasts


In [12]:
cat_counts = places_c["Category"].value_counts()
city_counts = places_c["City"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(cat_counts.index, cat_counts.values)
axes[0].set_title("Tourism Spot Categories")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(city_counts.index, city_counts.values)
axes[1].set_title("Spots by City")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(OUT / "eda_categories_cities.png", dpi=140)
plt.show()

# Nature = Cagar Alam (nature reserve)
nature = places_c[places_c["Category"] == "Cagar Alam"]
nature_by_city = nature.groupby("City").agg(
    n_spots=("Place_Id", "count"),
    avg_place_rating=("Rating", "mean"),
).sort_values(["n_spots", "avg_place_rating"], ascending=False)
print("Nature (Cagar Alam) by city:\n", nature_by_city)

# Combine volume + quality: score = n_spots * avg_rating
nature_by_city["score"] = nature_by_city["n_spots"] * nature_by_city["avg_place_rating"]
best_nature_city = nature_by_city["score"].idxmax()
print(f"\nBest city for nature enthusiasts: {best_nature_city}")
print(nature_by_city.loc[best_nature_city])


Nature (Cagar Alam) by city:
             n_spots  avg_place_rating
City                                 
Bandung          54          4.394444
Yogyakarta       23          4.482609
Semarang         20          4.325000
Surabaya          5          4.340000
Jakarta           4          4.375000

Best city for nature enthusiasts: Bandung
n_spots              54.000000
avg_place_rating      4.394444
score               237.300000
Name: Bandung, dtype: float64


### 2.4 Combine places + ratings — most loved spots/cities/categories


In [13]:
combo = ratings_c.merge(
    places_c[["Place_Id", "Place_Name", "Category", "City", "Price", "Rating"]],
    on="Place_Id",
    how="inner",
)
print("Combined shape:", combo.shape)

loved_places = (
    combo.groupby(["Place_Id", "Place_Name", "City", "Category"], as_index=False)
    .agg(avg_user_rating=("Place_Ratings", "mean"), n_ratings=("Place_Ratings", "count"))
)
# Prefer places with enough ratings
loved_places["love_score"] = loved_places["avg_user_rating"] * np.log1p(loved_places["n_ratings"])
top_places = loved_places.sort_values("love_score", ascending=False).head(10)
print("\nMost loved spots:\n", top_places.to_string(index=False))

loved_cities = combo.groupby("City").agg(
    avg_user_rating=("Place_Ratings", "mean"),
    n_ratings=("Place_Ratings", "count"),
).sort_values("avg_user_rating", ascending=False)
print("\nMost loved cities (by avg user rating):\n", loved_cities)

loved_cat = combo.groupby("Category").agg(
    avg_user_rating=("Place_Ratings", "mean"),
    n_ratings=("Place_Ratings", "count"),
).sort_values("avg_user_rating", ascending=False)
print("\nMost liked category:\n", loved_cat)
most_liked_category = loved_cat.index[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(loved_cat.index, loved_cat["avg_user_rating"].values)
ax.set_title("Average User Rating by Category")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(OUT / "eda_liked_categories.png", dpi=140)
plt.show()

top_places.to_csv(OUT / "most_loved_places.csv", index=False)
loved_cities.to_csv(OUT / "most_loved_cities.csv")
loved_cat.to_csv(OUT / "most_liked_categories.csv")


Combined shape: (9597, 8)

Most loved spots:
  Place_Id                                Place_Name       City      Category  avg_user_rating  n_ratings  love_score
      416                          Keraton Surabaya   Surabaya        Budaya         4.000000         28   13.469183
      322                               Bukit Jamur    Bandung    Cagar Alam         3.785714         28   12.747620
      300                         Sanghyang Heuleut    Bandung    Cagar Alam         3.655172         29   12.431963
      134                      Desa Wisata Gamplong Yogyakarta Taman Hiburan         3.620690         29   12.314680
      279         Masjid Agung Trans Studio Bandung    Bandung Tempat Ibadah         3.642857         28   12.266578
      401                            Taman Keputran   Surabaya Taman Hiburan         3.576923         26   11.788955
      437 Gereja Perawan Maria Tak Berdosa Surabaya   Surabaya Tempat Ibadah         3.333333         33   11.754535
       91         

### 2.5 Collaborative filtering recommender (sklearn NearestNeighbors on user–item matrix)

Given a place name, find similar places via item–item CF (cosine similarity on rating vectors).


In [14]:
# User-item matrix
ui = ratings_c.pivot_table(index="User_Id", columns="Place_Id", values="Place_Ratings", aggfunc="mean").fillna(0)
# Item vectors: places x users
item_matrix = ui.T.values  # (n_places, n_users)
place_ids = ui.columns.to_numpy()
id_to_name = places_c.set_index("Place_Id")["Place_Name"].to_dict()
name_to_id = {v: k for k, v in id_to_name.items()}

nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(item_matrix)

def recommend_places(place_name: str, n: int = 5):
    # fuzzy-ish exact / contains match
    if place_name not in name_to_id:
        matches = [n for n in name_to_id if place_name.lower() in n.lower()]
        if not matches:
            raise ValueError(f"Place not found: {place_name}")
        place_name = matches[0]
    pid = name_to_id[place_name]
    idx = int(np.where(place_ids == pid)[0][0])
    dists, inds = nn.kneighbors(item_matrix[idx:idx+1], n_neighbors=n + 1)
    recs = []
    for d, i in zip(dists[0], inds[0]):
        rid = int(place_ids[i])
        rname = id_to_name.get(rid, str(rid))
        if rid == pid:
            continue
        meta = places_c.loc[places_c["Place_Id"] == rid].iloc[0]
        recs.append({
            "Place_Id": rid,
            "Place_Name": rname,
            "City": meta["City"],
            "Category": meta["Category"],
            "cosine_distance": float(d),
            "similarity": float(1 - d),
        })
        if len(recs) >= n:
            break
    return place_name, pd.DataFrame(recs)

sample_place = "Monumen Nasional"
resolved, rec_df = recommend_places(sample_place, n=5)
print(f"Because you liked: {resolved}")
print(rec_df.to_string(index=False))
rec_df.to_csv(OUT / "sample_recommendations.csv", index=False)

# Second sample
resolved2, rec_df2 = recommend_places("Kota Tua", n=5)
print(f"\nBecause you liked: {resolved2}")
print(rec_df2.to_string(index=False))

part2_summary = {
    "n_users": int(users_c.shape[0]),
    "n_ratings": int(ratings_c.shape[0]),
    "n_places": int(places_c.shape[0]),
    "best_nature_city": str(best_nature_city),
    "most_liked_category": str(most_liked_category),
    "sample_seed_place": resolved,
    "sample_recommendations": rec_df.to_dict(orient="records"),
}
with open(OUT / "part2_summary.json", "w") as f:
    json.dump(part2_summary, f, indent=2)
print("\nSaved part2_summary.json")


Because you liked: Monumen Nasional
 Place_Id               Place_Name       City      Category  cosine_distance  similarity
      349    Wisata Mangrove Tapak   Semarang    Cagar Alam         0.728746    0.271254
      362        Danau Rawa Pening   Semarang    Cagar Alam         0.737271    0.262729
      118 Museum Sonobudoyo Unit I Yogyakarta        Budaya         0.739829    0.260171
        3            Dunia Fantasi    Jakarta Taman Hiburan         0.743764    0.256236
      318          Situ Patenggang    Bandung    Cagar Alam         0.758832    0.241168

Because you liked: Kota Tua
 Place_Id               Place_Name       City      Category  cosine_distance  similarity
      104            Tebing Breksi Yogyakarta        Budaya         0.718684    0.281316
      373   Museum Kereta Ambarawa   Semarang        Budaya         0.722983    0.277017
      107        Bangsal Pagelaran Yogyakarta        Budaya         0.767825    0.232175
       99 Kampung Wisata Kadipaten Yogyakarta

## Summary


In [15]:
print("=== PART 1 ===")
print(json.dumps(metrics, indent=2))
print("\n=== PART 2 ===")
print(json.dumps({k: v for k, v in part2_summary.items() if k != "sample_recommendations"}, indent=2))
print("\nSample recommendations for", part2_summary["sample_seed_place"])
print(pd.DataFrame(part2_summary["sample_recommendations"]).to_string(index=False))
print("\nArtifacts in:", OUT)
print("Models in:", MODELS)


=== PART 1 ===
{
  "img_size": 160,
  "batch": 32,
  "max_per_class_train_subset": 350,
  "num_classes": 10,
  "classes": [
    "altar",
    "apse",
    "bell_tower",
    "column",
    "dome(inner)",
    "dome(outer)",
    "flying_buttress",
    "gargoyle",
    "stained_glass",
    "vault"
  ],
  "no_aug_best_val_accuracy": 0.9253224730491638,
  "aug_best_val_accuracy": 0.9137814044952393,
  "final_best_val_accuracy": 0.9253224730491638,
  "final_best_val_loss": 0.2249237298965454,
  "improvement_aug_minus_no_aug": -0.01154106855392456,
  "best_checkpoint": "best_heritage_classifier.keras",
  "saved_model": "heritage_structure_classifier_best.keras"
}

=== PART 2 ===
{
  "n_users": 300,
  "n_ratings": 9597,
  "n_places": 437,
  "best_nature_city": "Bandung",
  "most_liked_category": "Taman Hiburan",
  "sample_seed_place": "Monumen Nasional"
}

Sample recommendations for Monumen Nasional
 Place_Id               Place_Name       City      Category  cosine_distance  similarity
      349  